In [0]:
df_annotated = (spark.table(T_VENTAS_RAW)
    .withColumn("err_cliente_null", F.col("id_cliente").isNull())
    .withColumn("err_qty_invalid", (F.col("cantidad").isNull()) | (F.col("cantidad") <= 0))
    .withColumn("err_price_invalid", (F.col("precio_unitario").isNull()) | (F.col("precio_unitario") <= 0))
    .withColumn("err_subtotal_mismatch",
        F.abs(F.col("subtotal_linea") -
              ((F.col("cantidad") * F.col("precio_unitario")) - (F.col("descuento_unitario") * F.col("cantidad"))))
        > F.lit(0.01))
    .withColumn("rejection_reasons", F.concat_ws(";",
        F.when(F.col("err_cliente_null"), F.lit("CUSTOMER_ID_NULL")),
        F.when(F.col("err_qty_invalid"), F.lit("INVALID_QUANTITY")),
        F.when(F.col("err_price_invalid"), F.lit("INVALID_PRICE")),
        F.when(F.col("err_subtotal_mismatch"), F.lit("SUBTOTAL_MISMATCH")),
    ))
)
 
w = Window.partitionBy("id_detalle_venta").orderBy(F.col("id_venta"))
df_deduped = df_annotated.withColumn("_rn", F.row_number().over(w))
 
drop_cols = ["_rn", "err_cliente_null", "err_qty_invalid", "err_price_invalid", "err_subtotal_mismatch"]
df_ventas_clean = (df_deduped.filter((F.length("rejection_reasons") == 0) & (F.col("_rn") == 1))
                    .drop(*drop_cols))
df_ventas_quar  = (df_deduped.filter((F.length("rejection_reasons") > 0) | (F.col("_rn") > 1))
                    .withColumn("quarantined_at", F.current_timestamp())
                    .drop(*drop_cols))
 
df_ventas_clean.write.mode("overwrite").saveAsTable(T_VENTAS_CL)
df_ventas_quar.write.mode("overwrite").saveAsTable(T_VENTAS_QT)
 
print(f"Silver ventas_clean: {spark.table(T_VENTAS_CL).count()} | ventas_quarantine: {spark.table(T_VENTAS_QT).count()}")
display(spark.table(T_VENTAS_QT).groupBy("rejection_reasons").count().orderBy(F.desc("count")))